In [ ]:
import { load } from "dotenv";
const env = await load();

const process = { env };
// process.env;


In [2]:
import { PromptTemplate } from "@langchain/core/prompts";
import { ChatOpenAI } from "@langchain/openai";
import { pull } from "langchain/hub";
import { createReactAgent, AgentExecutor } from "langchain/agents";
import { SerpAPI } from "@langchain/community/tools/serpapi";
import { Calculator } from "@langchain/community/tools/calculator";
import { DynamicTool } from "langchain/tools";


const prompt = await pull<PromptTemplate>("hwchase17/react");
  console.log(prompt);

const llm = new ChatOpenAI({
  temperature: 0,
  model: process.env.MODEL_NAME,
  configuration: {
    baseURL: process.env.BASE_URL,
    apiKey: process.env.OPENAI_API_KEY,
  },
});

// const dynamicTool = new DynamicTool({
//   name: 'Google Search',
//   description: 'A tool that returns the length of a given string',
//   func: async (input: string) => {
//     console.log(input);
//     return '5 Chinese Yuan'
//   },
//   returnDirect: true,
// });

const tools = [new SerpAPI(process.env.SERP_API_KEY), new Calculator()];

console.log(tools);

const agent = await createReactAgent({
  llm,
  tools,
  prompt,
});

const agentExecutor = new AgentExecutor({
  agent,
  tools,
});

const result = await agentExecutor.invoke({
  input: "我有 17 美元，现在相当于多少人民币？",
});

console.log(result);



PromptTemplate {
  lc_serializable: true,
  lc_kwargs: {
    template: "Answer the following questions as best you can. You have access to the following tools:\n" +
      "\n" +
      "{tools}\n" +
      "\n" +
      "Use the following format:\n" +
      "\n" +
      "Question: the input question you must answer\n" +
      "Thought: you should always think about what to do\n" +
      "Action: the action to take, should be one of [{tool_names}]\n" +
      "Action Input: the input to the action\n" +
      "Observation: the result of the action\n" +
      "... (this Thought/Action/Action Input/Observation can repeat N times)\n" +
      "Thought: I now know the final answer\n" +
      "Final Answer: the final answer to the original input question\n" +
      "\n" +
      "Begin!\n" +
      "\n" +
      "Question: {input}\n" +
      "Thought:{agent_scratchpad}",
    inputVariables: [ "agent_scratchpad", "input", "tool_names", "tools" ],
    templateFormat: "f-string",
    partialVariables: {